In [1]:
import pandas as pd

pd.set_option("display.max_columns", 200)

In [2]:
cases = pd.read_csv("../data/processed/cases.csv")
vdem = pd.read_csv("../data/raw/VDEMv16SelectedVars_data.csv")

In [3]:
cases["year"] = cases["year"].astype(int)
vdem["year"] = vdem["year"].astype(int)

In [4]:
vdem_hcname = vdem[["country_name", "year", "v2juhcname"]].copy()

In [5]:
cases = cases.merge(
    vdem_hcname,
    left_on=["country_clean", "year"],
    right_on=["country_name", "year"],
    how="left",
    validate="many_to_one"
)

In [6]:
cases = cases.drop(columns=["country_name_y"], errors="ignore")
cases = cases.rename(columns={"country_name_x": "country_name"})

In [7]:
lag_source = vdem_hcname.rename(columns={"country_name": "country_clean"}).copy()

In [8]:
for lag in [1, 2]:
    temp = lag_source.copy()
    temp["year"] += lag
    temp = temp.rename(columns={"v2juhcname": f"v2juhcname_lag{lag}"})

    cases = cases.merge(
        temp,
        on=["country_clean", "year"],
        how="left",
        validate="many_to_one"
    )

In [10]:
print(cases.shape)
print("Missing current:", cases["v2juhcname"].isna().sum())
print("Missing lag1:", cases["v2juhcname_lag1"].isna().sum())
print("Missing lag2:", cases["v2juhcname_lag2"].isna().sum())

print(
    cases.loc[cases["v2juhcname"].isna(), "country"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

(1077, 268)
Missing current: 5
Missing lag1: 5
Missing lag2: 5
['European Union', 'Namibia', 'Somalia']


In [11]:
cases[cases["country"].isin(["Namibia", "Somalia"])][
    ["country", "country_clean", "year", "v2juhcname"]
].sort_values(["country", "year"])

,country,country_clean,year,v2juhcname
193,Namibia,Namibia,1986,NaN
194,Namibia,Namibia,2010,Supreme Court of Namibia
195,Namibia,Namibia,2015,Supreme Court of Namibia
196,Namibia,Namibia,2019,Supreme Court of Namibia
251,Somalia,Somalia,2022,NaN
631,Somalia,Somalia,2022,NaN


In [13]:
vdem[vdem["country_name"].isin(["Namibia", "Somalia"])][
    ["country_name", "year", "v2juhcname"]
].sort_values(["country_name", "year"])

,country_name,year,v2juhcname
18599,Namibia,1900,NaN
18600,Namibia,1901,NaN
18601,Namibia,1902,NaN
18602,Namibia,1903,NaN
18603,Namibia,1904,NaN
...,...,...,...
19017,Somalia,2021,NaN
19018,Somalia,2022,NaN
19019,Somalia,2023,NaN
19020,Somalia,2024,NaN


In [15]:
cases.loc[cases["country"].isin(["Namibia", "Somalia"]), "country_clean"].unique()

array(['Namibia', 'Somalia'], dtype=object)

In [16]:
output_path = "../data/processed/cases_final.csv"

cases.to_csv(output_path, index=False)

print("Saved to:", output_path)

Saved to: ../data/processed/cases_final.csv


In [18]:
old = pd.read_csv("../data/processed/cases.csv")
new = pd.read_csv("../data/processed/cases_final.csv")

print("Old shape:", old.shape)
print("New shape:", new.shape)
print("Column difference:", new.shape[1] - old.shape[1])

new_only = sorted(set(new.columns) - set(old.columns))
old_only = sorted(set(old.columns) - set(new.columns))

print("New-only columns:", new_only)
print("Old-only columns:", old_only)

Old shape: (1077, 265)
New shape: (1077, 268)
Column difference: 3
New-only columns: ['v2juhcname', 'v2juhcname_lag1', 'v2juhcname_lag2']
Old-only columns: []
